# Knowledge Base Generation Pipeline

**Thesis:** RAG-Driven Natural Language Energy Demand Forecasting  
**Author:** Zoheb Anwar Hussain (Student ID: 1196931)  
**Model:** Gemini 3 Flash Preview (`gemini-3-flash-preview`)

---

### What this notebook does
Transforms raw time-series energy data from two datasets (GEFCom2012 and UCI Household)
into structured natural language summaries suitable for RAG retrieval. Each summary
describes a specific time period and granularity level, written in a human-friendly
tone grounded strictly in statistical evidence.

### Architecture — Thin orchestration layer
All logic lives in `src/knowledge_base/`. This notebook imports those functions and
calls them visibly cell-by-cell, making the pipeline easy to follow during viva
demonstration. The notebook contains no implementation — only orchestration.

### Pipeline Flow
```
Raw CSV Data (data/gefcom/, data/household/)
    ↓
Load + clean + reshape
    ↓
Aggregate (daily / weekly / monthly / seasonal / system_level / appliance / yearly)
    ↓
Validate (remove zero/null primary metric rows)
    ↓
Stratified sampling (50 per type, evenly across zones/years)
    ↓
Build prompts (Gemini-friendly templates)
    ↓
Generate summaries via Gemini 3 Flash
    ↓
Quality validation per summary
    ↓
Build master KB CSV with metadata + parent_id links
```

### Summary Types (10 total)
| Dataset | Types |
|---------|-------|
| GEFCom | daily, weekly, monthly, seasonal, system_level |
| Household | daily, weekly, monthly, appliance, yearly |

---
## Cell 1 — Imports and Setup

All logic is imported from `src/knowledge_base/`. Configuration is imported from `config/`.
Logging utilities are imported from `src/utils/`.

In [2]:
# Auto-reload modules so code changes in src/ take effect without notebook restart
%load_ext autoreload
%autoreload 2

# ── Configuration imports ────────────────────────────────────────────────────
from config import (
    PATHS,
    KB_MODEL_NAME,
    MAX_SUMMARIES_PER_TYPE,
)
from config.paths import create_all_directories

# ── Utility imports ──────────────────────────────────────────────────────────
from src.utils import setup_logger, log_section

# ── KB pipeline imports ──────────────────────────────────────────────────────
from src.knowledge_base import (
    # Data loading
    verify_data_paths,
    load_gefcom_data,
    load_household_data,
    # GEFCom aggregation
    reshape_gefcom_load,
    compute_gefcom_daily_stats,
    compute_gefcom_weekly_stats,
    compute_gefcom_monthly_stats,
    compute_gefcom_seasonal_stats,
    compute_gefcom_system_level,
    # Household aggregation
    clean_household,
    aggregate_household,
    compute_household_appliance,
    compute_household_yearly,
    # Validation
    validate_aggregates,
    # Prompt building
    build_gefcom_daily_prompts,
    build_gefcom_weekly_prompts,
    build_gefcom_monthly_prompts,
    build_gefcom_seasonal_prompts,
    build_gefcom_system_level_prompts,
    build_household_daily_prompts,
    build_household_weekly_prompts,
    build_household_monthly_prompts,
    build_household_appliance_prompts,
    build_household_yearly_prompts,
    # Generation
    configure_gemini_kb,
    generate_summaries,
    # Master KB builder
    build_master_knowledge_base,
)

import pandas as pd

# ── Initialise logger and create all output directories ──────────────────────
logger = setup_logger("kb_pipeline")
create_all_directories()

logger.info("All imports successful.")
logger.info("Model: %s", KB_MODEL_NAME)
logger.info("Pilot size per type: %d", MAX_SUMMARIES_PER_TYPE)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


2026-04-27 21:06:59 | INFO     | kb_pipeline | All imports successful.
2026-04-27 21:06:59 | INFO     | kb_pipeline | Model: gemini-3-flash-preview
2026-04-27 21:06:59 | INFO     | kb_pipeline | Pilot size per type: 50


---
## Cell 2 — Configure Gemini Client

Reads `GEMINI_API_KEY` from your `.env` file (or prompts for it interactively if missing).

In [3]:
gemini_client = configure_gemini_kb()

---
## Cell 3 — Load Raw Datasets

Loads GEFCom CSV files (with `thousands=','` to handle comma separators)
and the UCI Household power consumption file (with auto-detected delimiter).

In [5]:
log_section("Loading raw datasets", 1, 7)

verify_data_paths(
    gefcom_path=PATHS["data_gefcom"],
    household_path=PATHS["data_household"],
)

gefcom_frames = load_gefcom_data(PATHS["data_gefcom"])
household_df  = load_household_data(PATHS["data_household"])

logger.info(
    "Raw data loaded — GEFCom files: %d | Household rows: %d",
    len(gefcom_frames), len(household_df),
)


  STEP 1/7  [█░░░░░░] 14%
  Loading raw datasets



2026-04-27 21:10:15 | INFO     | kb_pipeline | Raw data loaded — GEFCom files: 6 | Household rows: 2075259


---
## Cell 4 — GEFCom Processing

Reshape, aggregate at all granularities, validate, and save to disk.

In [6]:
log_section("Processing GEFCom data", 2, 7)

if "load_history" in gefcom_frames:

    # Reshape wide → long
    gefcom_long = reshape_gefcom_load(gefcom_frames["load_history"])

    # Aggregate at all granularities
    gefcom_daily        = compute_gefcom_daily_stats(gefcom_long)
    gefcom_weekly       = compute_gefcom_weekly_stats(gefcom_daily)
    gefcom_monthly      = compute_gefcom_monthly_stats(gefcom_daily)
    gefcom_seasonal     = compute_gefcom_seasonal_stats(gefcom_daily)
    gefcom_system_level = compute_gefcom_system_level(
        gefcom_daily, gefcom_weekly, gefcom_monthly
    )

    # Validate — remove zero/null primary metric rows
    gefcom_daily    = validate_aggregates(gefcom_daily,    "gefcom_daily")
    gefcom_weekly   = validate_aggregates(gefcom_weekly,   "gefcom_weekly")
    gefcom_monthly  = validate_aggregates(gefcom_monthly,  "gefcom_monthly")
    gefcom_seasonal = validate_aggregates(gefcom_seasonal, "gefcom_seasonal")

    # Persist intermediate processed files
    gefcom_daily.to_csv(
        PATHS["proc_gefcom"] / "cleaned_daily.csv", index=False
    )
    gefcom_weekly.to_csv(
        PATHS["proc_gefcom"] / "weekly_aggregates.csv", index=False
    )
    gefcom_monthly.to_csv(
        PATHS["proc_gefcom"] / "monthly_aggregates.csv", index=False
    )
    gefcom_seasonal.to_csv(
        PATHS["proc_gefcom"] / "seasonal_aggregates.csv", index=False
    )
    for gran, df in gefcom_system_level.items():
        if not df.empty:
            df.to_csv(
                PATHS["proc_gefcom"] / f"system_level_{gran}.csv",
                index=False,
            )

    logger.info(
        "GEFCom processing complete — daily: %d | weekly: %d | "
        "monthly: %d | seasonal: %d | system-level daily: %d",
        len(gefcom_daily), len(gefcom_weekly), len(gefcom_monthly),
        len(gefcom_seasonal),
        len(gefcom_system_level.get("daily", pd.DataFrame())),
    )
else:
    logger.warning(
        "'load_history' not found in GEFCom files — skipping."
    )
    gefcom_daily        = pd.DataFrame()
    gefcom_weekly       = pd.DataFrame()
    gefcom_monthly      = pd.DataFrame()
    gefcom_seasonal     = pd.DataFrame()
    gefcom_system_level = {
        "daily":   pd.DataFrame(),
        "weekly":  pd.DataFrame(),
        "monthly": pd.DataFrame(),
    }


  STEP 2/7  [██░░░░░] 28%
  Processing GEFCom data



2026-04-27 21:10:32 | INFO     | kb_pipeline | GEFCom processing complete — daily: 31740 | weekly: 4680 | monthly: 1080 | seasonal: 380 | system-level daily: 1587


---
## Cell 5 — Household Processing

Clean minute-level data, resample to daily/weekly/monthly,
compute appliance shares, derive yearly aggregates with peak season.

In [7]:
log_section("Processing Household data", 3, 7)

household_clean   = clean_household(household_df)
household_daily   = aggregate_household(household_clean, freq="D",  label="daily")
household_weekly  = aggregate_household(household_clean, freq="W",  label="weekly")
household_monthly = aggregate_household(household_clean, freq="ME", label="monthly")

household_appliance = compute_household_appliance(
    household_daily, household_weekly, household_monthly
)
household_yearly = compute_household_yearly(household_monthly)

# Validate — remove zero/null primary metric rows
household_daily   = validate_aggregates(household_daily,   "household_daily")
household_weekly  = validate_aggregates(household_weekly,  "household_weekly")
household_monthly = validate_aggregates(household_monthly, "household_monthly")

# Persist intermediate processed files
household_clean.to_csv(
    PATHS["proc_household"] / "cleaned_minute.csv", index=False
)
household_daily.to_csv(
    PATHS["proc_household"] / "daily_aggregates.csv", index=False
)
household_weekly.to_csv(
    PATHS["proc_household"] / "weekly_aggregates.csv", index=False
)
household_monthly.to_csv(
    PATHS["proc_household"] / "monthly_aggregates.csv", index=False
)
for gran, df in household_appliance.items():
    if not df.empty:
        df.to_csv(
            PATHS["proc_household"] / f"appliance_{gran}.csv", index=False
        )
if not household_yearly.empty:
    household_yearly.to_csv(
        PATHS["proc_household"] / "yearly_aggregates.csv", index=False
    )

logger.info(
    "Household processing complete — daily: %d | weekly: %d | "
    "monthly: %d | yearly: %d | appliance daily: %d",
    len(household_daily), len(household_weekly), len(household_monthly),
    len(household_yearly),
    len(household_appliance.get("daily", pd.DataFrame())),
)


  STEP 3/7  [███░░░░] 42%
  Processing Household data



2026-04-27 21:11:06 | INFO     | kb_pipeline | Household processing complete — daily: 1433 | weekly: 207 | monthly: 48 | yearly: 5 | appliance daily: 1433


---
## Cell 6 — Build Prompt Inputs

Construct prompt-input rows using stratified sampling.
GEFCom samples evenly across `zone_id` (20 zones).
Household samples evenly across `year` (2006–2010).

In [8]:
log_section("Building prompt inputs", 4, 7)

LIMIT = MAX_SUMMARIES_PER_TYPE  # 50 per type

# ── GEFCom prompts ───────────────────────────────────────────────────────────
gefcom_daily_prompts    = build_gefcom_daily_prompts(gefcom_daily,       limit=LIMIT)
gefcom_weekly_prompts   = build_gefcom_weekly_prompts(gefcom_weekly,     limit=LIMIT)
gefcom_monthly_prompts  = build_gefcom_monthly_prompts(gefcom_monthly,   limit=LIMIT)
gefcom_seasonal_prompts = build_gefcom_seasonal_prompts(gefcom_seasonal, limit=LIMIT)
gefcom_system_prompts   = build_gefcom_system_level_prompts(
    gefcom_system_level, limit=LIMIT
)

# ── Household prompts ────────────────────────────────────────────────────────
household_daily_prompts     = build_household_daily_prompts(household_daily,     limit=LIMIT)
household_weekly_prompts    = build_household_weekly_prompts(household_weekly,   limit=LIMIT)
household_monthly_prompts   = build_household_monthly_prompts(household_monthly, limit=LIMIT)
household_appliance_prompts = build_household_appliance_prompts(
    household_appliance, limit=LIMIT
)
household_yearly_prompts    = build_household_yearly_prompts(household_yearly,   limit=LIMIT)

# Combine into two master DataFrames for inspection
gefcom_prompts = pd.concat([
    gefcom_daily_prompts, gefcom_weekly_prompts, gefcom_monthly_prompts,
    gefcom_seasonal_prompts, gefcom_system_prompts,
], ignore_index=True)

household_prompts = pd.concat([
    household_daily_prompts, household_weekly_prompts,
    household_monthly_prompts, household_appliance_prompts,
    household_yearly_prompts,
], ignore_index=True)

# Persist prompt-input CSVs for inspection / audit
gefcom_prompts.to_csv(
    PATHS["prompt_inputs"] / "gefcom_prompt_inputs.csv", index=False
)
household_prompts.to_csv(
    PATHS["prompt_inputs"] / "household_prompt_inputs.csv", index=False
)

logger.info(
    "Prompt inputs built — GEFCom: %d | Household: %d | Total: %d",
    len(gefcom_prompts), len(household_prompts),
    len(gefcom_prompts) + len(household_prompts),
)


  STEP 4/7  [████░░░] 57%
  Building prompt inputs



2026-04-27 21:11:14 | INFO     | kb_pipeline | Prompt inputs built — GEFCom: 208 | Household: 182 | Total: 390


---
## Cell 7 — Generate Summaries via Gemini 3 Flash

Each `generate_summaries()` call is **idempotent** — if the output CSV already
contains rows from a previous run, those rows are skipped automatically and
only missing rows are generated.

Estimated runtime (50 rows per type × 10 types × ~9s per row ≈ **70 minutes total**).

In [9]:
log_section("Generating summaries via Gemini 3 Flash", 5, 7)

# Log file paths shared across all 10 generation calls
request_log = PATHS["logs"] / "gemini_requests_log.csv"
failure_log = PATHS["logs"] / "failed_rows.csv"

# ── GEFCom (5 types) ─────────────────────────────────────────────────────────
logger.info("=" * 55)
logger.info("Generating GEFCom summaries (5 types)")
logger.info("=" * 55)

gefcom_daily_summaries = generate_summaries(
    client=gemini_client,
    prompt_df=gefcom_daily_prompts,
    output_path=PATHS["summaries_csv"] / "gefcom_daily_summaries.csv",
    request_log_path=request_log,
    failure_log_path=failure_log,
)
logger.info(
    "GEFCom daily complete: %d summaries.", len(gefcom_daily_summaries)
)

gefcom_weekly_summaries = generate_summaries(
    client=gemini_client,
    prompt_df=gefcom_weekly_prompts,
    output_path=PATHS["summaries_csv"] / "gefcom_weekly_summaries.csv",
    request_log_path=request_log,
    failure_log_path=failure_log,
)
logger.info(
    "GEFCom weekly complete: %d summaries.", len(gefcom_weekly_summaries)
)

gefcom_monthly_summaries = generate_summaries(
    client=gemini_client,
    prompt_df=gefcom_monthly_prompts,
    output_path=PATHS["summaries_csv"] / "gefcom_monthly_summaries.csv",
    request_log_path=request_log,
    failure_log_path=failure_log,
)
logger.info(
    "GEFCom monthly complete: %d summaries.", len(gefcom_monthly_summaries)
)

gefcom_seasonal_summaries = generate_summaries(
    client=gemini_client,
    prompt_df=gefcom_seasonal_prompts,
    output_path=PATHS["summaries_csv"] / "gefcom_seasonal_summaries.csv",
    request_log_path=request_log,
    failure_log_path=failure_log,
)
logger.info(
    "GEFCom seasonal complete: %d summaries.", len(gefcom_seasonal_summaries)
)

gefcom_system_summaries = generate_summaries(
    client=gemini_client,
    prompt_df=gefcom_system_prompts,
    output_path=PATHS["summaries_csv"] / "gefcom_system_level_summaries.csv",
    request_log_path=request_log,
    failure_log_path=failure_log,
)
logger.info(
    "GEFCom system-level complete: %d summaries.", len(gefcom_system_summaries)
)

# ── Household (5 types) ──────────────────────────────────────────────────────
logger.info("=" * 55)
logger.info("Generating Household summaries (5 types)")
logger.info("=" * 55)

household_daily_summaries = generate_summaries(
    client=gemini_client,
    prompt_df=household_daily_prompts,
    output_path=PATHS["summaries_csv"] / "household_daily_summaries.csv",
    request_log_path=request_log,
    failure_log_path=failure_log,
)
logger.info(
    "Household daily complete: %d summaries.", len(household_daily_summaries)
)

household_weekly_summaries = generate_summaries(
    client=gemini_client,
    prompt_df=household_weekly_prompts,
    output_path=PATHS["summaries_csv"] / "household_weekly_summaries.csv",
    request_log_path=request_log,
    failure_log_path=failure_log,
)
logger.info(
    "Household weekly complete: %d summaries.", len(household_weekly_summaries)
)

household_monthly_summaries = generate_summaries(
    client=gemini_client,
    prompt_df=household_monthly_prompts,
    output_path=PATHS["summaries_csv"] / "household_monthly_summaries.csv",
    request_log_path=request_log,
    failure_log_path=failure_log,
)
logger.info(
    "Household monthly complete: %d summaries.",
    len(household_monthly_summaries),
)

household_appliance_summaries = generate_summaries(
    client=gemini_client,
    prompt_df=household_appliance_prompts,
    output_path=PATHS["summaries_csv"] / "household_appliance_summaries.csv",
    request_log_path=request_log,
    failure_log_path=failure_log,
)
logger.info(
    "Household appliance complete: %d summaries.",
    len(household_appliance_summaries),
)

household_yearly_summaries = generate_summaries(
    client=gemini_client,
    prompt_df=household_yearly_prompts,
    output_path=PATHS["summaries_csv"] / "household_yearly_summaries.csv",
    request_log_path=request_log,
    failure_log_path=failure_log,
)
logger.info(
    "Household yearly complete: %d summaries.",
    len(household_yearly_summaries),
)

# Cumulative count
total = sum([
    len(gefcom_daily_summaries),     len(gefcom_weekly_summaries),
    len(gefcom_monthly_summaries),   len(gefcom_seasonal_summaries),
    len(gefcom_system_summaries),    len(household_daily_summaries),
    len(household_weekly_summaries), len(household_monthly_summaries),
    len(household_appliance_summaries), len(household_yearly_summaries),
])
logger.info("All summary generation complete. Total: %d summaries.", total)

2026-04-27 21:11:25 | INFO     | kb_pipeline | =======================================================
2026-04-27 21:11:25 | INFO     | kb_pipeline | Generating GEFCom summaries (5 types)
2026-04-27 21:11:25 | INFO     | kb_pipeline | =======================================================



  STEP 5/7  [█████░░] 71%
  Generating summaries via Gemini 3 Flash



gefcom_daily_summaries:   0%|          | 0/40 [00:00<?, ?it/s]

Summary rejected: too short (21 words, minimum 30).
Summary for 'gefcom_daily_1_2007-11-29' failed quality check — skipping.
Attempt 1/3 failed for 'gefcom_daily_2_2005-08-07': 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}
Attempt 2/3 failed for 'gefcom_daily_2_2005-08-07': 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}
Summary rejected: too short (20 words, minimum 30).
Summary for 'gefcom_daily_2_2005-08-07' failed quality check — skipping.
Summary rejected: too short (16 words, minimum 30).
Summary for 'gefcom_daily_2_2007-11-29' failed quality check — skipping.
Summary rejected: too short (22 words, minimum 30).
Summary for 'gefcom_daily_3_2007-11-29' failed quality check — skipping.
Summar

gefcom_weekly_summaries:   0%|          | 0/40 [00:00<?, ?it/s]

...

Summary rejected: too short (20 words, minimum 30).
Summary for 'household_monthly_2006-12-31' failed quality check — skipping.
Summary rejected: too short (20 words, minimum 30).
Summary for 'household_monthly_2007-11-30' failed quality check — skipping.
Summary rejected: too short (22 words, minimum 30).
Summary for 'household_monthly_2007-10-31' failed quality check — skipping.
Summary rejected: too short (20 words, minimum 30).
Summary for 'household_monthly_2007-01-31' failed quality check — skipping.
Attempt 1/3 failed for 'household_monthly_2007-09-30': 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}
Summary rejected: too short (21 words, minimum 30).
Summary for 'household_monthly_2007-09-30' failed quality check — skipping.
Summary rejected: too short (19 words, minimum 30).
Summary for 'household_monthly_2007-06-30' failed quality 

household_appliance_summaries:   0%|          | 0/43 [00:00<?, ?it/s]

Summary rejected: too short (28 words, minimum 30).
Summary for 'household_appliance_daily_2006-12-16' failed quality check — skipping.
Summary rejected: too short (23 words, minimum 30).
Summary for 'household_appliance_daily_2006-12-17' failed quality check — skipping.
Summary rejected: too short (23 words, minimum 30).
Summary for 'household_appliance_daily_2007-07-14' failed quality check — skipping.
Attempt 1/3 failed for 'household_appliance_daily_2007-02-03': 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}
Summary rejected: too short (22 words, minimum 30).
Summary for 'household_appliance_daily_2007-01-16' failed quality check — skipping.
Summary rejected: too short (18 words, minimum 30).
Summary for 'household_appliance_daily_2008-07-12' failed quality check — skipping.
Summary rejected: too short (20 words, minimum 30).
Summary fo

household_yearly_summaries:   0%|          | 0/5 [00:00<?, ?it/s]

Summary rejected: too short (23 words, minimum 30).
Summary for 'household_yearly_2008' failed quality check — skipping.
Attempt 1/3 failed for 'household_yearly_2010': 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}
Attempt 2/3 failed for 'household_yearly_2010': 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}
2026-04-28 00:53:51 | INFO     | kb_pipeline | Household yearly complete: 4 summaries.
2026-04-28 00:53:51 | INFO     | kb_pipeline | All summary generation complete. Total: 140 summaries.


---
## Cell 8 — Build Master KB CSV

Merges all 10 summary DataFrames into the single `combined_master_summaries.csv`
consumed by all downstream pipeline stages. Adds metadata columns and `parent_id`
links for hierarchical retrieval.

In [10]:
log_section("Building master Knowledge Base CSV", 6, 7)

master_kb = build_master_knowledge_base(
    summary_dfs=[
        gefcom_daily_summaries,        gefcom_weekly_summaries,
        gefcom_monthly_summaries,      gefcom_seasonal_summaries,
        gefcom_system_summaries,       household_daily_summaries,
        household_weekly_summaries,    household_monthly_summaries,
        household_appliance_summaries, household_yearly_summaries,
    ],
    output_path=PATHS["summaries_csv"] / "combined_master_summaries.csv",
)


  STEP 6/7  [██████░] 85%
  Building master Knowledge Base CSV



---
## Cell 9 — Validation Report

Quick summary of the final master KB — total entries, per-type breakdown,
metadata coverage, sample summaries (to verify human-friendly tone).

In [11]:
log_section("Validation report", 7, 7)

print("\n" + "=" * 65)
print("  KNOWLEDGE BASE PIPELINE — VALIDATION REPORT")
print("=" * 65)

if master_kb.empty:
    print("  Master KB is empty — check logs for errors.")
else:
    print(f"\n  Total KB entries : {len(master_kb)}")
    print(f"  Total columns    : {len(master_kb.columns)}")
    print(f"  Columns          : {list(master_kb.columns)}")

    # Per-type breakdown
    expected_types = [
        ("gefcom",    "daily"),       ("gefcom",    "weekly"),
        ("gefcom",    "monthly"),     ("gefcom",    "seasonal"),
        ("gefcom",    "system_level"),
        ("household", "daily"),       ("household", "weekly"),
        ("household", "monthly"),     ("household", "appliance"),
        ("household", "yearly"),
    ]
    print("\n  Summary type check (10 expected):")
    for dataset, granularity in expected_types:
        count = len(master_kb[
            (master_kb["dataset"] == dataset)
            & (master_kb["granularity"] == granularity)
        ])
        status = "OK" if count > 0 else "MISSING"
        print(
            f"    {dataset:<12} {granularity:<14} {count:>4} entries   {status}"
        )

    # Metadata column populations
    print("\n  Metadata column population:")
    for col in ["zone_id", "year", "season", "parent_id"]:
        if col in master_kb.columns:
            populated = (master_kb[col] != "").sum()
            pct = populated / len(master_kb) * 100
            print(
                f"    {col:<12} {populated:>4} / {len(master_kb)} "
                f"rows populated ({pct:.0f}%)"
            )

    # Timestamp format check
    if "generated_at" in master_kb.columns:
        sample_ts = master_kb["generated_at"].iloc[0]
        print("\n  Timestamp format check:")
        print(f"    Sample generated_at : '{sample_ts}'")
        expected_fmt = len("15-01-2024 14:30:22") == len(str(sample_ts))
        print(f"    DD-MM-YYYY HH:MM:SS : {'OK' if expected_fmt else 'CHECK'}")

    # Sample summaries
    print("\n" + "-" * 65)
    print("  SAMPLE SUMMARIES (verifying human-friendly tone)")
    print("-" * 65)
    for _, row in master_kb.sample(
        n=min(3, len(master_kb)), random_state=42
    ).iterrows():
        print(
            f"\n  [{row['dataset'].upper()} / {row['granularity']}] "
            f"{row['row_id']}"
        )
        preview = str(row["summary"])[:350]
        ellipsis = "..." if len(str(row["summary"])) > 350 else ""
        print(f"  {preview}{ellipsis}")

print("\n" + "=" * 65)


  STEP 7/7  [███████] 100%
  Validation report


  KNOWLEDGE BASE PIPELINE — VALIDATION REPORT

  Total KB entries : 140
  Total columns    : 12
  Columns          : ['kb_id', 'row_id', 'dataset', 'granularity', 'zone_id', 'year', 'month', 'season', 'parent_id', 'summary', 'context_json', 'generated_at']

  Summary type check (10 expected):
    gefcom       daily             9 entries   OK
    gefcom       weekly           16 entries   OK
    gefcom       monthly          35 entries   OK
    gefcom       seasonal         24 entries   OK
    gefcom       system_level     21 entries   OK
    household    daily             5 entries   OK
    household    weekly            8 entries   OK
    household    monthly           4 entries   OK
    household    appliance        14 entries   OK
    household    yearly            4 entries   OK

  Metadata column population:
    zone_id       105 / 140 rows populated (75%)
    year           86 / 140 rows populated (61%)
    season         24 / 140

---
## Pipeline Complete

The Knowledge Base is generated and saved to:
```
outputs/knowledge_base/generated_summaries/csv/combined_master_summaries.csv
```

### Next Phase
Move to `notebooks/02_golden_dataset.ipynb` to generate the golden evaluation dataset.